## 1. Import Data

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit

## 2. Load Data

In [10]:
DATA_PATH = '/workspace/data/metrics.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.set_index('timestamp').sort_index()

# Memory utilization %
df['mem_util_percent'] = (df['mem_used_gb'] / (df['mem_used_gb'] + df['mem_available_gb'])) * 100

df = df.dropna()

## 3. Make isolation forest

In [11]:
features = [
    'cpu_percent',
    'mem_util_percent',
    'gpu_util_percent',
    'gpu_mem_used_gb'
]

X = df[features].copy()

iso_forest = IsolationForest(
    contamination=0.03,
    random_state=42,
    n_estimators=100
)

df['anomaly_score'] = iso_forest.fit_predict(X)
df['anomaly'] = df['anomaly_score'] == -1

## 4. Feature Engineering

In [12]:
df['cpu_rolling_mean'] = df['cpu_percent'].rolling(window=10).mean()
df['mem_rolling_mean'] = df['mem_util_percent'].rolling(window=10).mean()
df['gpu_rolling_mean'] = df['gpu_util_percent'].rolling(window=10).mean()
df['gpu_mem_rolling_mean'] = df['gpu_mem_used_gb'].rolling(window=10).mean()

df = df.dropna()

rolling_features = [
    'cpu_rolling_mean',
    'mem_rolling_mean',
    'gpu_rolling_mean',
    'gpu_mem_rolling_mean'
]

X = df[rolling_features].copy()

y = df['anomaly']

## 5. Building the pipeline

In [13]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))

tscv = TimeSeriesSplit(n_splits=5)

results = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    results.append({
        'precision': precision_score(y_test, predictions),
        'recall': recall_score(y_test, predictions),
        'f1': f1_score(y_test, predictions)
        })

results_df = pd.DataFrame(results)


/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## 6. Summarize the results

In [14]:
results_df.mean()
results_df.std()

precision    0.162966
recall       0.438842
f1           0.236984
dtype: float64

## 7. Confusion Matrix

In [15]:
print(confusion_matrix(y_test, predictions))

[[1788   82]
 [  11    3]]


In [16]:
print(y.value_counts())

anomaly
False    10965
True       340
Name: count, dtype: int64


## Observations

- Notebook took our collected homelab data, prepped the data for using in a Logistic Regression pipeline, and trained the model to detect anomalies.
- Results showed the model performed poorly with the data set we had, originally finding all data was "normal" and not anomolous
- This was addressed by adding class_weight=balanced to the model to have it weigh missed anomalies more in the data to allow us to find them.  The final result was mixed.  A lower precision score (82 false positives) and a midling recall score (3 of 14 anomalies found) of the final run are an example of this.  Over all, the model only successfully found 44% of the true positives
- I believe this was primarily caused by the model needing supervised data and our anomaly label was generated by an unsupervised IsolationForest.  Models that require supervised data may underperform when given unsupervised data due mislabel and the lack of clear, defined outputs that supervised learning relies on during training
- Next steps should be to generate some supervised data for the model to compare how it affects the model's performance.  Additionally, it would be good to try a Random Forest as well